In [59]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split

In [60]:
colunas = [
    'top-left', 'top-middle', 'top-right',
    'middle-left', 'middle-middle', 'middle-right',
    'bottom-left', 'bottom-middle', 'bottom-right',
    'class'
]

df_raw = pd.read_csv(r"tic+tac+toe+endgame\tic-tac-toe.data", names = colunas)
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   top-left       958 non-null    str  
 1   top-middle     958 non-null    str  
 2   top-right      958 non-null    str  
 3   middle-left    958 non-null    str  
 4   middle-middle  958 non-null    str  
 5   middle-right   958 non-null    str  
 6   bottom-left    958 non-null    str  
 7   bottom-middle  958 non-null    str  
 8   bottom-right   958 non-null    str  
 9   class          958 non-null    str  
dtypes: str(10)
memory usage: 75.0 KB


In [61]:
mapeamento_tabuleiro = {'x': 1, 'o': -1, 'b': 0}
for col in colunas[:-1]:
    df_raw[col] = df_raw[col].map(mapeamento_tabuleiro)

In [62]:
def verificar_estado_completo(row):
    # Converte a linha (as 9 posições) em uma lista
    tab = row[:9].values
    
    # Definição das linhas, colunas e diagonais
    vitorias = [[0,1,2], [3,4,5], [6,7,8], [0,3,6], [1,4,7], [2,5,8], [0,4,8], [2,4,6]]
    
    # 1. Verifica Vencedores
    for v in vitorias:
        soma = tab[v[0]] + tab[v[1]] + tab[v[2]]
        if soma == 3: return "X vence"
        if soma == -3: return "O vence"
    
    # 2. Verifica Possibilidade de Fim de Jogo (alguém tem 2 marcas e a 3ª é vazia)
    for v in vitorias:
        soma = tab[v[0]] + tab[v[1]] + tab[v[2]]
        if abs(soma) == 2 and 0 in [tab[v[0]], tab[v[1]], tab[v[2]]]:
            return "Possibilidade de Fim de Jogo"
            
    # 3. Verifica Empate (tabuleiro cheio sem vencedor)
    if 0 not in tab:
        return "Empate"
        
    # 4. Caso contrário, o jogo ainda está rolando
    return "Tem jogo"

# Aplicando a nova rotulação no df_raw
df_raw['target'] = df_raw.apply(verificar_estado_completo, axis=1)

In [63]:
print(df_raw['target'].value_counts())

target
X vence    626
O vence    316
Empate      16
Name: count, dtype: int64


In [64]:
# 1. Verificar o que já temos no df_raw tratado
contagem_atual = df_raw['target'].value_counts().to_dict()
meta = 200
dados_complementares = []

# 2. Loop para gerar apenas as classes que faltam ou estão abaixo da meta
classes_necessarias = ["Tem jogo", "Possibilidade de Fim de Jogo", "Empate", "X vence", "O vence"]

for classe in classes_necessarias:
    vagas_sobrando = meta - contagem_atual.get(classe, 0)
    
    # Se a classe já tem mais de 200 (como 'X vence'), vamos apenas manter 200 depois
    if vagas_sobrando <= 0:
        continue
        
    print(f"Gerando {vagas_sobrando} novas amostras para: {classe}")
    
    count = 0
    while count < vagas_sobrando:
        # Gera tabuleiro aleatório: 1 (X), -1 (O), 0 (Vazio)
        tab = [random.choice([1, -1, 0]) for _ in range(9)]
        estado = verificar_estado_completo(pd.Series(tab)) # Usando sua função anterior
        
        if estado == classe:
            registro = {f"pos{i}": tab[i] for i in range(9)}
            # Ajustando nomes das colunas para bater com o df_raw
            colunas_fixas = {df_raw.columns[i]: tab[i] for i in range(9)}
            colunas_fixas['target'] = estado
            dados_complementares.append(colunas_fixas)
            count += 1

# 3. Combinar o df_raw original com os novos dados
df_complemento = pd.DataFrame(dados_complementares)
df_final = pd.concat([df_raw, df_complemento], ignore_index=True)

# 4. Ajuste Final: Garantir exatamente 200 de cada (Truncar excessos)
df_final = df_final.groupby('target').head(meta).reset_index(drop=True)

print("\nDistribuição Final do Dataset:")
print(df_final['target'].value_counts())

Gerando 200 novas amostras para: Tem jogo
Gerando 200 novas amostras para: Possibilidade de Fim de Jogo
Gerando 184 novas amostras para: Empate

Distribuição Final do Dataset:
target
X vence                         200
O vence                         200
Empate                          200
Tem jogo                        200
Possibilidade de Fim de Jogo    200
Name: count, dtype: int64


In [65]:
df_final = df_final.drop(columns=["class"])

In [66]:
x = df_final.drop(columns=["target"])
y = df_final["target"]

In [67]:
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.30, random_state=42, stratify=y)

In [68]:
x_test, x_val, y_test, y_val = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

In [69]:
train_df = pd.concat([x_train, y_train], axis=1)
val_df = pd.concat([x_val, y_val], axis=1)
test_df = pd.concat([x_test, y_test], axis=1)

In [70]:
train_df.to_csv("train_df.csv", index = False)
val_df.to_csv("val_df.csv", index = False)
test_df.to_csv("test_df.csv", index = False)